In [11]:
#!/usr/bin/env python3
"""
Friedman test + Wilcoxon-Holm post-hoc + Critical Difference diagrams
for feature-transformation comparison, per classifier and per metric.

Input : <dataset>_output_roc.csv and <dataset>_output_pr.csv (one pair per dataset)
        rows    = feature transformations
        columns = classifiers
Output: 10 analyses (2 metrics x 5 classifiers), each = one Friedman test over
        30 datasets (blocks) x 9 transformations (treatments), plus a CD diagram.

Multiple-testing correction (two separate levels, both Holm):
  LEVEL 1 (across tests)  : Holm over the 10 Friedman p-values. A post-hoc is run
                            only for tests whose Holm-adjusted Friedman p < alpha
                            (use --posthoc-all to run it for every test anyway).
  LEVEL 2 (within a test) : Holm over all 36 pairwise Wilcoxon signed-rank tests
                            of that one test.

AUC values are used as-is (higher = better, rank 1 = best). Ties get average ranks.

Requires: numpy, pandas, scipy, matplotlib
"""
import argparse
import itertools
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, rankdata, wilcoxon

DEFAULT_DIR = (r"C:\Users\Shehroz.Khan\OneDrive - Al-Arabia Educational Enterprises Company"
               r"\OCC - Categorical Data\Output files - With Minority Test Class")
METRICS = {"roc": "AUC ROC", "pr": "AUC PR"}



In [13]:



# ----------------------------------------------------------------------------- stats
def holm(pvals):
    """Holm-Bonferroni adjusted p-values (same order as input)."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj_sorted = np.minimum(1.0, (m - np.arange(m)) * p[order])
    adj_sorted = np.maximum.accumulate(adj_sorted)
    adj = np.empty(m)
    adj[order] = adj_sorted
    return adj



In [15]:

def safe_wilcoxon(x, y):
    d = np.asarray(x) - np.asarray(y)
    if np.all(d == 0):
        return 1.0
    return float(wilcoxon(x, y, zero_method="wilcox", alternative="two-sided").pvalue)

In [17]:
# ----------------------------------------------------------------------------- data
def load_metric(folder: Path, metric: str):
    files = sorted(folder.glob(f"*_output_{metric}.csv"))
    if not files:
        sys.exit(f"No files matching *_output_{metric}.csv in {folder}")
    frames, names = [], []
    for f in files:
        df = pd.read_csv(f, index_col=0)
        if df.isna().any().any():
            sys.exit(f"NaN found in {f.name}")
        frames.append(df)
        names.append(f.name[: -len(f"_output_{metric}.csv")])
    ref = frames[0]
    for n, df in zip(names, frames):
        if set(df.index) != set(ref.index) or set(df.columns) != set(ref.columns):
            sys.exit(f"Row/column labels of dataset '{n}' differ from '{names[0]}'")
    # align order to the first file -> array [dataset, transformation, classifier]
    arr = np.stack([df.loc[ref.index, ref.columns].to_numpy(float) for df in frames])
    return names, list(ref.index), list(ref.columns), arr


In [19]:
# ----------------------------------------------------------------------------- CD diagram
def cliques_from_sig(sig, order):
    """Contiguous groups (in rank order) in which no pair differs significantly."""
    k = len(order)
    groups = []
    for i in range(k):
        j = i
        while j + 1 < k and all(not sig[order[a], order[j + 1]] for a in range(i, j + 1)):
            j += 1
        if j > i:
            groups.append((i, j))
    return [g for g in groups if not any(o != g and o[0] <= g[0] and g[1] <= o[1] for o in groups)]

In [21]:
def cd_diagram(mean_ranks, sig, title, subtitle, path):
    labels = list(mean_ranks.index)
    k = len(labels)
    order = list(np.argsort(mean_ranks.to_numpy()))          # best (lowest rank) first
    r = mean_ranks.to_numpy()
    groups = cliques_from_sig(sig, order)

    dy = 0.32
    bar_y0 = -0.45
    label_y0 = bar_y0 - 0.25 * max(len(groups), 1) - 0.35
    half = (k + 1) // 2
    fig_h = 2.2 + dy * half
    fig, ax = plt.subplots(figsize=(10.5, fig_h))

    lo, hi = 1, k
    ax.plot([lo, hi], [0, 0], color="black", lw=1.4)
    for t in range(lo, hi + 1):
        ax.plot([t, t], [0, 0.12], color="black", lw=1.2)
        ax.text(t, 0.2, str(t), ha="center", va="bottom", fontsize=10)
    ax.text((lo + hi) / 2, 0.75, "mean rank across datasets (1 = best AUC)", ha="center", fontsize=10)

    for n, idx in enumerate(order):
        left = n < half
        y = label_y0 - dy * (n if left else n - half)
        x_end = lo - 0.6 if left else hi + 0.6
        ax.plot([r[idx], r[idx], x_end], [0, y, y], color="0.25", lw=0.9)
        ax.text(x_end + (-0.08 if left else 0.08), y, f"{labels[idx]}  ({r[idx]:.2f})",
                ha="right" if left else "left", va="center", fontsize=10)

    for m, (i, j) in enumerate(groups):
        y = bar_y0 - 0.25 * m
        ax.plot([r[order[i]] - 0.04, r[order[j]] + 0.04], [y, y], color="crimson", lw=4,
                solid_capstyle="round", zorder=5)

    ax.set_xlim(lo - 3.4, hi + 3.4)
    ax.set_ylim(label_y0 - dy * half - 0.1, 1.2)
    ax.axis("off")
    fig.suptitle(title, fontsize=12, fontweight="bold", y=0.99)
    ax.set_title(subtitle, fontsize=9, color="0.3", pad=2)
    fig.tight_layout()
    fig.savefig(path.with_suffix(".png"), dpi=200, bbox_inches="tight")
    fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


In [23]:

# ----------------------------------------------------------------------------- main
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--input-dir", default=DEFAULT_DIR)
    ap.add_argument("--output-dir", default=None, help="default: <input-dir>/friedman_results")
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--posthoc-all", action="store_true",
                    help="run post-hoc for every test, ignoring the Level-1 gate")
    ap.add_argument("--expected-datasets", type=int, default=30)
    args, _ = ap.parse_known_args(argv)   # ignores extras such as Jupyter's "-f kernel.json"

    in_dir = Path(args.input_dir)
    out_dir = Path(args.output_dir) if args.output_dir else in_dir / "friedman_results"
    (out_dir / "figures").mkdir(parents=True, exist_ok=True)
    alpha = args.alpha

    # ---- load both metrics
    data = {}
    for metric in METRICS:
        data[metric] = load_metric(in_dir, metric)
    if data["roc"][0] != data["pr"][0]:
        sys.exit("Dataset names for ROC and PR files do not match.")
    n_ds = len(data["roc"][0])
    if n_ds != args.expected_datasets:
        print(f"WARNING: found {n_ds} datasets, expected {args.expected_datasets}")
    print(f"Datasets: {n_ds} | transformations: {len(data['roc'][1])} | "
          f"classifiers: {len(data['roc'][2])}")

    # ---- Friedman for all 10 (metric, classifier) pairs
    rows, rank_tables = [], {}
    for metric, label in METRICS.items():
        names, transf, clfs, arr = data[metric]
        for c, clf in enumerate(clfs):
            X = arr[:, :, c]                                   # datasets x transformations
            ranks = np.apply_along_axis(lambda v: rankdata(-v, method="average"), 1, X)
            mr = pd.Series(ranks.mean(axis=0), index=transf)
            stat, p = friedmanchisquare(*[X[:, t] for t in range(X.shape[1])])
            if np.isnan(p):
                stat, p = 0.0, 1.0
            rank_tables[(metric, clf)] = (X, mr)
            rows.append(dict(metric=label, classifier=clf, n_datasets=X.shape[0],
                             n_transformations=X.shape[1], friedman_chi2=stat, p_raw=p,
                             best_transformation=mr.idxmin(),
                             **{f"rank_{t}": mr[t] for t in transf}))
    summary = pd.DataFrame(rows)

    # ---- LEVEL 1 correction: Holm across the 10 omnibus tests
    summary["p_holm_across_10_tests"] = holm(summary["p_raw"].to_numpy())
    summary["omnibus_significant"] = summary["p_holm_across_10_tests"] < alpha
    summary["posthoc_run"] = summary["omnibus_significant"] | args.posthoc_all

    # ---- Post-hoc (Wilcoxon + LEVEL 2 Holm within each test) and CD diagrams
    pair_rows, n_sig_pairs, top_group = [], [], []
    for i, row in summary.iterrows():
        metric = [m for m, l in METRICS.items() if l == row["metric"]][0]
        clf = row["classifier"]
        X, mr = rank_tables[(metric, clf)]
        transf = list(mr.index)
        k = len(transf)
        sig = np.zeros((k, k), dtype=bool)

        if row["posthoc_run"]:
            pairs = list(itertools.combinations(range(k), 2))
            p_raw = [safe_wilcoxon(X[:, a], X[:, b]) for a, b in pairs]
            p_adj = holm(p_raw)
            for (a, b), pr, pa in zip(pairs, p_raw, p_adj):
                sig[a, b] = sig[b, a] = pa < alpha
                pair_rows.append(dict(metric=row["metric"], classifier=clf,
                                      transformation_A=transf[a], transformation_B=transf[b],
                                      p_raw=pr, p_holm_within_test=pa, significant=pa < alpha,
                                      better=transf[a] if mr.iloc[a] < mr.iloc[b] else transf[b]))
            status = "post-hoc: Wilcoxon, Holm within test (36 pairs)"
        else:
            status = "post-hoc NOT run: Friedman not significant after Holm across 10 tests"

        n_sig_pairs.append(int(np.triu(sig).sum()))
        order = list(np.argsort(mr.to_numpy()))
        # transformations statistically tied with the best-ranked one
        best = order[0]
        top_group.append(", ".join(transf[j] for j in order
                                   if j == best or not sig[best, j]) if row["posthoc_run"] else "")

        title = f"{row['metric']} - {clf}"
        sub = (f"Friedman chi2={row['friedman_chi2']:.2f}, p={row['p_raw']:.2g}, "
               f"Holm-adjusted across 10 tests p={row['p_holm_across_10_tests']:.2g} | "
               f"{status} | alpha={alpha}")
        cd_diagram(mr, sig, title, sub,
                   out_dir / "figures" / f"CD_{metric.upper()}_{clf}")

    summary["n_significant_pairs"] = n_sig_pairs
    summary["best_and_statistically_tied"] = top_group
    summary.to_csv(out_dir / "friedman_summary.csv", index=False)
    pd.DataFrame(pair_rows).to_csv(out_dir / "posthoc_pairwise.csv", index=False)

    # ---- console report
    print("\n" + "=" * 78)
    print(f"LEVEL 1: Holm correction applied ACROSS the 10 Friedman tests (alpha={alpha})")
    print("LEVEL 2: Holm correction applied WITHIN each test across its 36 pairwise tests")
    print("=" * 78)
    show = summary[["metric", "classifier", "friedman_chi2", "p_raw",
                    "p_holm_across_10_tests", "omnibus_significant",
                    "best_transformation", "n_significant_pairs"]]
    print(show.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print(f"\nResults written to: {out_dir}")




In [25]:
if __name__ == "__main__":
    main()

Datasets: 30 | transformations: 9 | classifiers: 5

LEVEL 1: Holm correction applied ACROSS the 10 Friedman tests (alpha=0.05)
LEVEL 2: Holm correction applied WITHIN each test across its 36 pairwise tests
 metric classifier  friedman_chi2     p_raw  p_holm_across_10_tests  omnibus_significant best_transformation  n_significant_pairs
AUC ROC  Euclidean          46.15 2.226e-07               1.558e-06                 True             OHE-MCA                    9
AUC ROC   Canberra          48.07 9.584e-08               7.667e-07                 True             OHE-MCA                    9
AUC ROC       ISOF          18.97   0.01503                 0.03007                 True                 OHE                    3
AUC ROC        LOF          21.38  0.006202                 0.01861                 True                 OHE                    0
AUC ROC       OSVM          30.31 0.0001862               0.0009311                 True                 OHE                    6
 AUC PR  Eucli